### Lab Assignment: Python Warmup and Logfile Analytics

### University of Virginia
### DS 7200: Distributed Computing
### Last Updated: August 20, 2023

---

This lab consists of two parts. Part 1 is the Python warmup. Part 2 is the logfile analytics in Spark.  
Answer the questions in this assignment, showing all code and solutions.

**Total points: 15**

---

### Part 1: Python Warmup

1) (1 PT) Rename this notebook to JupyterTutorial_[your_initials], where you will enter your initials in place of [your_initials].

2) (1 PT) In the cell below, enter a list of data science topics you find interesting.  Use the markdown style (you will need to change the style from the Code style).

Schema, Data Life Cycles, Architecture, Tokenization, NLP, Multimodal Machine Learning, Optimization

3) (1 PT) In the cell below, enter the following Python list:

some_vals = [-1, 6, 12, 34]  

You will use the Code style, run the cell, and print the list.

In [64]:
some_vals = [-1, 6, 12, 34]

print(some_vals)

[-1, 6, 12, 34]


4) (1 PT) Use a list comprehension to return a filtered list containing only the values greater than 6.  
Call this list *some_vals_filtered* and print it.

In [65]:
some_vals_filtered = [x for x in some_vals if x > 6]

print(some_vals_filtered)

[12, 34]


Next, a small pandas dataframe is constructed.

In [66]:
import pandas as pd

df = pd.DataFrame({'first_name': ['Andy','Crystal'],
                   'domain_facebook' : [1,1],
                   'domain_foursquare' : [0,0],
                   'age' : [20, 32]})
df

,first_name,domain_facebook,domain_foursquare,age
0,Andy,1,0,20
1,Crystal,1,0,32


5) (1 PT) In the cell below, write a list comprehension that returns the fields names in the dataframe `df` containing the string *domain*.  Run the cell to verify the correct result.

In [67]:
[x for x in df if "domain" in x]

['domain_facebook', 'domain_foursquare']

6) (1 PT) Use the list comprehension from (5) to index into `df` and show the data for columns containing *domain*

In [68]:
df[[x for x in df if "domain" in x]]

,domain_facebook,domain_foursquare
0,1,0
1,1,0


7) (1 PT) In the cell below, print the *domain_facebook* column

In [69]:
print(df["domain_facebook"])
# or
# print(df.loc[:, 'domain_facebook'])

0    1
1    1
Name: domain_facebook, dtype: int64


8) (1 PT) In the cell below, print the row with index 1.

In [70]:
# recall: iloc for integer based position index (start from 0) and loc for label based indexes

print(df.iloc[1])

first_name           Crystal
domain_facebook            1
domain_foursquare          0
age                       32
Name: 1, dtype: object


9) (1 PT) Next, you will cube the *age* column of `df` and assign the result to a new column called *agecube*.

Specifically, call the `apply` method with a `lambda function` inside to cube the *age* column.  
Print the dataframe.

In [71]:
df['agecube'] = df['age'].apply(lambda x: x ** 3)

print(df)

  first_name  domain_facebook  domain_foursquare  age  agecube
0       Andy                1                  0   20     8000
1    Crystal                1                  0   32    32768


10) (1 PT) Given the list of strings below, form one string, placing semicolons between each word.  It should look like this:  

`'the;quick;brown;fox'`

Print the resulting string.

In [72]:
some_list = ['the','quick','brown','fox']

In [73]:
print(";".join(some_list))

# Is this is better than map for this?

the;quick;brown;fox


### Part 2: Logfile Analytics

Import modules for Spark Session and regex 

In [74]:
from pyspark.sql import SparkSession
import re

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

11) (1 PT) Read in the logfile.txt data

In [75]:
# read in the data
lines = sc.textFile("logfile.txt")
type(lines)

pyspark.core.rdd.RDD

12) (1 PT) Count the number of rows of data

In [76]:
lines.count()

360

13) (1 PT) Show the first 5 lines

In [77]:
lines.take(5)

[' 01 ',
 '03/22 08:51:01 INFO   :.main: *************** RSVP Agent started ***************',
 ' 02 ',
 '03/22 08:51:01 INFO   :...locate_configFile: Specified configuration file: /u/user10/rsvpd1.conf',
 '03/22 08:51:01 INFO   :.main: Using log level 511']

14) (1 PT) Show all lines containing the word WARNING and write code to count them

In [78]:
warning_lines = lines.filter(lambda line: "WARNING" in line)
print(warning_lines.collect()) # .collect() returns a list of all elements in the RDD

print(f"\nThere are {warning_lines.count()} lines containing 'WARNING'.")

['03/22 08:51:06 WARNING:.....mailslot_create: setsockopt(MCAST_ADD) failed - EDC8116I Address not available.', '03/22 08:51:06 WARNING:.....mailslot_create: setsockopt(MCAST_ADD) failed - EDC8116I Address not available.', '03/22 08:51:06 WARNING:.....mailslot_create: setsockopt(MCAST_ADD) failed - EDC8116I Address not available.', '03/22 08:51:06 WARNING:.....mailslot_create: setsockopt(MCAST_ADD) failed - EDC8116I Address not available.']

There are 4 lines containing 'WARNING'.


15) (1 PT) Write a word count program to count the number of each of these log levels:  

terms = 'WARNING|INFO|EVENT|PROTERR|TRACE'

In [79]:
def count_logs(line):
    # count 'WARNING'
    warning_count = lines.filter(lambda line: "WARNING" in line).count()
    # count 'INFO'
    info_count = lines.filter(lambda line: "INFO" in line).count()
    # count 'EVENT'
    event_count = lines.filter(lambda line: "EVENT" in line).count()
    # count 'PROTERR'
    proterr_count = lines.filter(lambda line: "PROTERR" in line).count()
    # count 'TRACE'
    trace_count = lines.filter(lambda line: "TRACE" in line).count()
    return (warning_count, info_count, event_count, proterr_count, trace_count)

In [80]:
count_logs(lines)

(4, 145, 13, 1, 119)